# 02. 글자 단위 Transformer 태거 (GPU)
- 입력: 글자 + 초/중/종성 임베딩, 출력: 원본 초/중/종성 3-헤드
- 후보 마스크: 노이즈 채널 역방향 후보만 허용
- 노이즈 증강: 채널 확률표(받침·연음 결합)로 train output 을 매 epoch 새로 난독화
- 검증 분할은 01 노트북과 동일 (seed 42, 10%) → 점수 직접 비교 가능

**런타임 유형을 GPU(T4 이상)로 설정.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/데이콘/난독화
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1) 증강 샘플 확인 (실제 난독화와 비슷한지 눈으로 체크)

In [ ]:
import pandas as pd, random
from noisy_channel import JamoChannel, aligned_rows
from neural_tagger import NoiseSampler
tr = pd.read_csv('data/train.csv')
s = NoiseSampler(JamoChannel(aligned_rows(tr)))
random.seed(0)
for i in range(5):
    print('원본 :', tr.output[i][:80])
    print('실제 :', tr.input[i][:80])
    print('증강 :', s.corrupt(tr.output[i].rstrip())[:80]); print()

## 2) 빠른 동작 확인 (2 epoch, val 200행)

In [ ]:
!python neural_tagger.py --mode train --epochs 2 --eval_every 1 --val_limit 200

## 3) 본 학습 (train 90% → val 10% 로 best 저장)
T4 기준 epoch 당 1분 안팎 예상. 시간이 부족하면 `--epochs 15`.

In [ ]:
!python neural_tagger.py --mode train --epochs 30

## 4) 오류 분석 (01 노트북 결과와 비교)

In [ ]:
from collections import Counter
v = pd.read_csv('outputs/val_pred_tagger.csv')
print('tagger val F1', v.f1.mean())
err = Counter((a, b) for g, p in zip(v.output, v.pred) for a, b in zip(g, p) if a != b)
print(err.most_common(30))
for _, r in v.sort_values('f1').head(5).iterrows():
    print('IN  :', r.input[:120]); print('GT  :', r.output[:120]); print('PRED:', r.pred[:120]); print()

## 5) test 예측 → `outputs/submission_tagger.csv`
val 로 고른 best 체크포인트 사용. (val 까지 포함해 다시 학습하려면 `--full --epochs <best epoch>` 후 `--ckpt outputs/tagger_full.pt`)

In [ ]:
!python neural_tagger.py --mode predict
pd.read_csv('outputs/submission_tagger.csv').head()